<a href="https://colab.research.google.com/github/Andyfer004/HT1-RL/blob/main/Tak2_HT1_RL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CC3104 – Aprendizaje por Refuerzo
## Hoja de Trabajo 1 – Task 2

**Integrantes:** Andy Fuentes 22944, Diederich Solís 22952 y Diego Linares 221256.

Se ejecutan cinco episodios por configuración con semilla 42 para obtener resultados reproducibles.

## Configuración del entorno

In [1]:
import random
import numpy as np

# GridWorld original: S_t es la posición actual y A_t es una dirección.
GRID_SIZE = 4
GOAL_STATE = (3, 3)
BASE_TRAPS = {(1, 1), (2, 3)}
ACTIONS = {
    0: (-1, 0),  # arriba
    1: (1, 0),   # abajo
    2: (0, -1),  # izquierda
    3: (0, 1),   # derecha
}


class GridWorldEnv:
    def __init__(self, trap_states=BASE_TRAPS, step_reward=-1.0):
        self.trap_states = set(trap_states)
        self.step_reward = step_reward

    def reset(self):
        self.state = (0, 0)
        self.done = False
        return self.state

    def step(self, action):
        dr, dc = ACTIONS[action]
        r, c = self.state
        next_state = (
            max(0, min(GRID_SIZE - 1, r + dr)),
            max(0, min(GRID_SIZE - 1, c + dc)),
        )

        # R_{t+1} depende del estado alcanzado por la transición.
        if next_state == GOAL_STATE:
            reward, self.done = 10.0, True
        elif next_state in self.trap_states:
            reward, self.done = -5.0, True
        else:
            reward = self.step_reward

        self.state = next_state
        return next_state, reward, self.done


class RandomAgent:
    # Política estocástica uniforme: pi(a|s) = 1/4.
    def select_action(self, state):
        return random.randint(0, len(ACTIONS) - 1)


def run_episode(env, agent, gamma, max_steps=50):
    state = env.reset()
    rewards = []

    for _ in range(max_steps):
        action = agent.select_action(state)
        state, reward, done = env.step(action)
        rewards.append(reward)
        if done:
            break

    # Fórmula de clase: G_t = R_{t+1} + gamma * G_{t+1}.
    G = 0.0
    for reward in reversed(rewards):
        G = reward + gamma * G

    if state == GOAL_STATE:
        terminal = "meta"
    elif state in env.trap_states:
        terminal = f"trampa {state}"
    else:
        terminal = "límite"
    return G, len(rewards), terminal


def run_experiment(gamma=0.95, trap_states=BASE_TRAPS,
                   step_reward=-1.0, n_episodes=5, seed=42):
    # Reiniciar la semilla permite comparar configuraciones justamente.
    random.seed(seed)
    np.random.seed(seed)
    env = GridWorldEnv(trap_states, step_reward)
    agent = RandomAgent()
    return [run_episode(env, agent, gamma) for _ in range(n_episodes)]


def print_summary(label, results):
    returns = [r[0] for r in results]
    steps = [r[1] for r in results]
    terminals = [r[2] for r in results]
    print(label)
    print("Retornos G0:", [round(x, 4) for x in returns])
    print(f"Promedio: {np.mean(returns):.4f} | Mínimo: {min(returns):.4f} | Máximo: {max(returns):.4f}")
    print("Pasos:", steps)
    print("Finales:", terminals)


## 1. Cambio de $\gamma$ a 0.5 y 0.99

In [2]:
results_05 = run_experiment(gamma=0.5)
results_099 = run_experiment(gamma=0.99)
print_summary("GAMMA = 0.5", results_05)
print()
print_summary("GAMMA = 0.99", results_099)

GAMMA = 0.5
Retornos G0: [-2.0117, -2.0, -2.0, -2.0, -2.0007]
Promedio: -2.0025 | Mínimo: -2.0117 | Máximo: -2.0000
Pasos: [9, 22, 50, 17, 13]
Finales: ['trampa (1, 1)', 'trampa (1, 1)', 'meta', 'trampa (2, 3)', 'trampa (1, 1)']

GAMMA = 0.99
Retornos G0: [-12.3393, -23.0759, -32.7771, -19.1115, -15.7934]
Promedio: -20.6194 | Mínimo: -32.7771 | Máximo: -12.3393
Pasos: [9, 22, 50, 17, 13]
Finales: ['trampa (1, 1)', 'trampa (1, 1)', 'meta', 'trampa (2, 3)', 'trampa (1, 1)']


### Análisis del cambio en $\gamma$

| $\gamma$ | Retorno promedio | Retorno mínimo | Retorno máximo |
|:---:|---:|---:|---:|
| $0.50$ | $-2.0025$ | $-2.0117$ | $-2.0000$ |
| $0.99$ | $-20.6194$ | $-32.7771$ | $-12.3393$ |

> **Observación.** El retorno promedio disminuyó de $-2.0025$ con $\gamma=0.5$ a $-20.6194$ con $\gamma=0.99$, aunque se ejecutaron las mismas acciones gracias al uso de la misma semilla. Con $\gamma=0.5$, las recompensas futuras se descuentan rápidamente y $G_0$ depende principalmente de las primeras transiciones; con $\gamma=0.99$, las penalizaciones de trayectorias largas conservan casi todo su peso. Por ello, el episodio de 50 pasos obtuvo el retorno más negativo con $\gamma=0.99$, aunque finalmente alcanzó la meta.

## 2. Nueva trampa en (0, 3)

In [3]:
baseline_traps = run_experiment(gamma=0.95, trap_states=BASE_TRAPS)
new_traps = run_experiment(gamma=0.95, trap_states=BASE_TRAPS | {(0, 3)})
print_summary("TRAMPAS ORIGINALES", baseline_traps)
print()
print_summary("CON NUEVA TRAMPA (0, 3)", new_traps)

TRAMPAS ORIGINALES
Retornos G0: [-10.0487, -14.8916, -17.5702, -13.3981, -11.8946]
Promedio: -13.5606 | Mínimo: -17.5702 | Máximo: -10.0487
Pasos: [9, 22, 50, 17, 13]
Finales: ['trampa (1, 1)', 'trampa (1, 1)', 'meta', 'trampa (2, 3)', 'trampa (1, 1)']

CON NUEVA TRAMPA (0, 3)
Retornos G0: [-10.0487, -14.8916, -17.5702, -13.3981, -7.1394]
Promedio: -12.6096 | Mínimo: -17.5702 | Máximo: -7.1394
Pasos: [9, 22, 50, 17, 4]
Finales: ['trampa (1, 1)', 'trampa (1, 1)', 'meta', 'trampa (2, 3)', 'trampa (0, 3)']


### Análisis de la nueva trampa en $(0,3)$

| Configuración | Retorno promedio | Retorno mínimo | Retorno máximo |
|:---|---:|---:|---:|
| Trampas originales | $-13.5606$ | $-17.5702$ | $-10.0487$ |
| Nueva trampa en $(0,3)$ | $-12.6096$ | $-17.5702$ | $-7.1394$ |

> **Observación.** Al agregar la trampa en $(0,3)$, el retorno promedio aumentó de $-13.5606$ a $-12.6096$ y el retorno máximo pasó de $-10.0487$ a $-7.1394$. La modificación afectó únicamente al quinto episodio, que terminó en la nueva trampa después de 4 pasos, mientras que los otros cuatro retornos permanecieron iguales. Aunque el resultado parece contradictorio, terminar rápidamente permitió acumular menos penalizaciones de $-1$, lo cual indica que la recompensa de $-5$ quizá no penaliza suficientemente la caída en una trampa.

## 3. Recompensa por paso de -1 a -0.1

In [4]:
step_m1 = run_experiment(gamma=0.95, step_reward=-1.0)
step_m01 = run_experiment(gamma=0.95, step_reward=-0.1)
print_summary("RECOMPENSA POR PASO = -1", step_m1)
print()
print_summary("RECOMPENSA POR PASO = -0.1", step_m01)

RECOMPENSA POR PASO = -1
Retornos G0: [-10.0487, -14.8916, -17.5702, -13.3981, -11.8946]
Promedio: -13.5606 | Mínimo: -17.5702 | Máximo: -10.0487
Pasos: [9, 22, 50, 17, 13]
Finales: ['trampa (1, 1)', 'trampa (1, 1)', 'meta', 'trampa (2, 3)', 'trampa (1, 1)']

RECOMPENSA POR PASO = -0.1
Retornos G0: [-3.9903, -3.0217, -1.0281, -3.3204, -3.6211]
Promedio: -2.9963 | Mínimo: -3.9903 | Máximo: -1.0281
Pasos: [9, 22, 50, 17, 13]
Finales: ['trampa (1, 1)', 'trampa (1, 1)', 'meta', 'trampa (2, 3)', 'trampa (1, 1)']


### Análisis de la recompensa por paso

| Recompensa por paso | Retorno promedio | Retorno mínimo | Retorno máximo |
|:---:|---:|---:|---:|
| $-1.0$ | $-13.5606$ | $-17.5702$ | $-10.0487$ |
| $-0.1$ | $-2.9963$ | $-3.9903$ | $-1.0281$ |

> **Observación.** Al cambiar la recompensa por paso de $-1$ a $-0.1$, el retorno promedio aumentó de $-13.5606$ a $-2.9963$, porque las transiciones no terminales fueron diez veces menos costosas. La cantidad de pasos y los estados terminales no cambiaron, ya que el agente sigue una política aleatoria fija y no utiliza las recompensas para actualizar su comportamiento. En un agente que sí aprendiera, esta modificación reduciría la presión por finalizar rápidamente y podría favorecer trayectorias más largas pero seguras, dependiendo de las recompensas terminales.